# 08 - RecBole Evaluation and LLM Comparison

This notebook trains the best RecBole CF models (from HPO) on full data,
evaluates on test, and compares with LLM results.

**Data split (from notebook 06):**
- `train.inter`: `user_liked` for ALL users + `recommended_accepted` for train users
- `valid.inter`: `recommended_accepted` for valid users (early stopping)
- `test.inter`: `recommended_accepted` for test users (evaluation)

This mirrors LLM evaluation where `user_liked` is provided as prompt context
and `recommended_accepted` is the prediction target.

**Two Method Categories:**

| Category | Description |
|----------|-------------|
| **Standalone** | CF models rank among full catalog (~6,924 items) |
| **CBF+Reranker** | Content-based retrieval (250 candidates) → reranker (CF or LLM) |

The LLM pipeline is inherently a two-stage approach: CBF retrieval selects candidates,
then the LLM reranks. For fair comparison, CF models are also evaluated as rerankers
on the same candidate pools.

**Models (CF only):**
- Pop, Random, BPR, ItemKNN, EASE, LightGCN, NeuMF

**Outputs:**
- `data/recbole/evaluation_results/recbole_llm_comparison.csv`
- `data/recbole/evaluation_results/comparison_summary.csv`
- `data/recbole/evaluation_results/recbole_final_results.json`
- Visualization PNGs

In [ ]:
# PyTorch 2.x compatibility fix for RecBole checkpoints
# Must be run BEFORE importing RecBole
# Uses a flag to prevent double-patching on re-runs
import torch

if not getattr(torch, "_recbole_patched", False):
    _original_load = torch.load

    def _patched_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _original_load(*args, **kwargs)

    torch.load = _patched_load
    torch._recbole_patched = True
    print("Applied weights_only=False patch for RecBole compatibility")
else:
    print("PyTorch patch already applied (skipping)")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm.auto import tqdm

    print(
        "WARNING: tqdm.notebook not available; install ipywidgets for proper notebook progress bars."
    )

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed

warnings.filterwarnings("ignore")

# Configuration
DATA_PATH = Path("../data")
RECBOLE_DATA_PATH = DATA_PATH / "recbole"
HPO_RESULTS_PATH = RECBOLE_DATA_PATH / "hpo_results"
RESULTS_PATH = RECBOLE_DATA_PATH / "evaluation_results"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

SEED = 42
K_VALUES = [1, 5, 10]

In [ ]:
def get_device() -> str:
    """Get best available device: CUDA > CPU (RecBole does not support MPS)."""
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


DEVICE = get_device()
print(f"Using device: {DEVICE}")

## Load Best Hyperparameters and Mappings

In [ ]:
# Load best hyperparameters from HPO
with open(HPO_RESULTS_PATH / "best_hyperparameters.json") as f:
    best_hyperparams = json.load(f)

print("Loaded best hyperparameters for:")
for model_name in best_hyperparams:
    print(f"  - {model_name}")

In [ ]:
# Load evaluation targets and ID mapping
with open(RECBOLE_DATA_PATH / "redial" / "evaluation_targets.json") as f:
    targets_data = json.load(f)

test_targets = {int(k): v for k, v in targets_data["test_targets"].items()}
test_user_ids = [int(uid) for uid in targets_data["test_user_ids"]]

with open(RECBOLE_DATA_PATH / "redial" / "id_to_title.json") as f:
    id_to_title = {int(k): v for k, v in json.load(f).items()}

dialogue_to_user = {
    str(k): int(v) for k, v in targets_data.get("test_dialogue_to_user", {}).items()
}

print(f"Loaded {len(test_targets)} test users with targets")
print(f"Loaded {len(dialogue_to_user)} dialogue_id mappings for content-based matching")
print(f"Loaded {len(id_to_title)} item ID to title mappings")

In [ ]:
# Sanity check: user/interactions counts
import csv
from math import ceil


def _count_users_and_interactions(p):
    users = set()
    n = 0
    with open(p, newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        user_key = [k for k in reader.fieldnames if k.startswith("user_id")][0]
        for row in reader:
            users.add(row[user_key])
            n += 1
    return len(users), n


train_path = RECBOLE_DATA_PATH / "redial" / "redial.train.inter"
valid_path = RECBOLE_DATA_PATH / "redial" / "redial.valid.inter"
test_path = RECBOLE_DATA_PATH / "redial" / "redial.test.inter"

train_users, train_n = _count_users_and_interactions(train_path)
valid_users, valid_n = _count_users_and_interactions(valid_path)
test_users, test_n = _count_users_and_interactions(test_path)

print("Split counts:")
print(
    f"  Train: {train_n} interactions, {train_users} users (user_liked for all + recommended_accepted for train)"
)
print(
    f"  Valid: {valid_n} interactions, {valid_users} users (recommended_accepted for valid)"
)
print(
    f"  Test:  {test_n} interactions, {test_users} users (recommended_accepted for test)"
)

# Verify test users appear in train (warm-start)
train_user_set = set()
with open(train_path, newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    user_key = [k for k in reader.fieldnames if k.startswith("user_id")][0]
    for row in reader:
        train_user_set.add(row[user_key])

test_user_set = set()
with open(test_path, newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    user_key = [k for k in reader.fieldnames if k.startswith("user_id")][0]
    for row in reader:
        test_user_set.add(row[user_key])

test_in_train = test_user_set & train_user_set
print(
    f"\nWarm-start check: {len(test_in_train)}/{len(test_user_set)} test users appear in train (should be all)"
)

# Expected per-epoch training batches (approx)
train_batches = ceil(train_n / 256)
print(f"Expected training batches per epoch (train / 256): ~{train_batches}")

## Model Definitions and Configuration

In [ ]:
# Model categories (CF only - sequential models excluded due to RecBole bug #1593)
# See: https://github.com/RUCAIBox/RecBole/issues/1593
GENERAL_CF_MODELS = ["Pop", "Random", "BPR", "ItemKNN", "EASE", "LightGCN", "NeuMF"]
ALL_MODELS = GENERAL_CF_MODELS

print(f"Models to evaluate: {len(ALL_MODELS)}")
print(f"CF models: {GENERAL_CF_MODELS}")

In [ ]:
def get_model_config(model_name: str, best_params: dict) -> dict:
    """Build configuration dictionary with best hyperparameters for final training."""
    config = {
        # Dataset settings — use proper 3-way split from notebook 06
        # train.inter: user_liked for ALL users + recommended_accepted for train users
        # valid.inter: recommended_accepted for valid users (early stopping)
        # test.inter:  recommended_accepted for test users (evaluation)
        "data_path": str(RECBOLE_DATA_PATH),
        "dataset": "redial",
        "benchmark_filename": ["train", "valid", "test"],
        # Field definitions
        "USER_ID_FIELD": "user_id",
        "ITEM_ID_FIELD": "item_id",
        "TIME_FIELD": "timestamp",
        "load_col": {
            "inter": ["user_id", "item_id", "timestamp"],
        },
        # Sequence settings
        "MAX_ITEM_LIST_LENGTH": 50,
        # Training settings
        "epochs": 100,
        "train_batch_size": 256,
        "eval_batch_size": 256,
        "learning_rate": 0.001,
        "stopping_step": 10,
        # Evaluation settings
        "eval_args": {
            "group_by": "user",
            "order": "TO",
            "split": {"LS": "valid_and_test"},
            "mode": "full",
        },
        "metrics": ["Recall", "MRR", "NDCG", "Hit", "Precision"],
        "topk": K_VALUES,
        "valid_metric": "NDCG@10",
        "train_neg_sample_args": None,
        # Device and reproducibility
        "device": DEVICE,
        "seed": SEED,
        "reproducibility": True,
        "show_progress": False,
        # Logging
        "log_wandb": False,
        "state": "INFO",
        # Best hyperparameters from HPO
        **best_params,
    }

    # Enable negative sampling for BPR-loss models
    if model_name in ["BPR", "LightGCN", "NeuMF"]:
        config["train_neg_sample_args"] = {
            "distribution": "uniform",
            "sample_num": 1,
            "dynamic": False,
        }

    return config

In [ ]:
# Clear RecBole dataset cache to prevent stale state from previous runs
# RecBole caches processed datasets to disk; stale cache can cause
# "not enough values to unpack (expected 3, got 2)" errors
import shutil

for cache_dir in ["dataset", "log", "saved"]:
    if Path(cache_dir).exists():
        shutil.rmtree(cache_dir)
        print(f"Cleared {cache_dir}/")
    else:
        print(f"{cache_dir}/ not found (OK)")

Path("saved").mkdir(exist_ok=True)
print("RecBole cache cleared.")

## Train and Evaluate All Models

In [ ]:
def train_and_evaluate(model_name: str, config_dict: dict) -> dict:
    """Train model on train split and evaluate on test set.

    Returns dict with status, metrics, AND model/trainer/dataset for reuse.
    """
    try:
        config = Config(model=model_name, config_dict=config_dict)
        init_seed(config["seed"], config["reproducibility"])

        dataset = create_dataset(config)
        train_data, valid_data, test_data = data_preparation(config, dataset)

        model = get_model(config["model"])(config, train_data._dataset).to(
            config["device"]
        )
        trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

        # Train with early stopping on valid (recommended_accepted for valid users)
        best_valid_score, best_valid_result = trainer.fit(
            train_data, valid_data, verbose=False, show_progress=False
        )

        # Evaluate on test (recommended_accepted for test users)
        test_result = trainer.evaluate(test_data)

        return {
            "status": "success",
            "best_valid_score": float(best_valid_score),
            "test_result": {k: float(v) for k, v in test_result.items()},
            # Store for filtered evaluation (avoids retraining)
            "model": model,
            "trainer": trainer,
            "dataset": train_data._dataset,
        }
    except Exception as e:
        print(f"  Error: {e}")
        return {
            "status": "failed",
            "error": str(e),
        }

In [ ]:
# Train and evaluate all models
recbole_results = {}

for model_name in tqdm(ALL_MODELS, desc="Training models"):
    print(f"\n{'=' * 60}")
    print(f"Training {model_name}")
    print(f"{'=' * 60}")

    hp = best_hyperparams.get(model_name, {})
    best_params = hp.get("params", {})

    config_dict = get_model_config(model_name, best_params)
    config_dict["model"] = model_name

    result = train_and_evaluate(model_name, config_dict)
    recbole_results[model_name] = result

    if result["status"] == "success":
        print(f"\n{model_name} Test Results:")
        for metric, value in result["test_result"].items():
            if "@10" in metric:
                print(f"  {metric}: {value:.4f}")

In [ ]:
# Save final trained models (best HPs, trained on train+valid)
# These are the models used for evaluation — save for reproducibility
FINAL_MODELS_PATH = RECBOLE_DATA_PATH / "final_models"
FINAL_MODELS_PATH.mkdir(parents=True, exist_ok=True)

saved_count = 0
for model_name, result in recbole_results.items():
    if result["status"] != "success":
        print(f"  Skipping {model_name} (training failed)")
        continue
    model_path = FINAL_MODELS_PATH / f"{model_name}.pth"
    torch.save(result["model"].state_dict(), model_path)
    size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f"  Saved {model_name} → {model_path} ({size_mb:.1f} MB)")
    saved_count += 1

print(f"\nSaved {saved_count} final models to {FINAL_MODELS_PATH}")

## Standalone and CBF+Reranker Evaluation

**Standalone evaluation**: Score all items per user using `recommendation_metrics()`.
This replaces RecBole's internal `trainer.evaluate()` for consistency.

**CBF+Reranker evaluation**: Load candidate sets used for LLM evaluation and restrict
predictions to the same candidate pool. All methods (CF rerankers and LLM rerankers)
use the same `recommendation_metrics()` function for direct comparability.

In [ ]:
import re

from stability.metrics import recommendation_metrics

# =============================================================================
# ALIGNMENT VERIFICATION
# =============================================================================
# Verify that candidate files and test_user_ids have matching counts
# This is critical for index-based matching to work correctly

print("=" * 60)
print("ALIGNMENT VERIFICATION")
print("=" * 60)

CANDIDATE_SIZES = [250, 500, 1000]
alignment_ok = True

for n_cand in CANDIDATE_SIZES:
    filepath = DATA_PATH / "processed" / f"test_prompt_examples_c{n_cand}_r10.jsonl"
    if filepath.exists():
        with open(filepath) as f:
            candidate_count = sum(1 for _ in f)
        print(f"  Candidates {n_cand}: {candidate_count} records")
        if candidate_count != len(test_user_ids):
            print(f"    WARNING: Mismatch with test_user_ids ({len(test_user_ids)})")
            alignment_ok = False

print(f"  test_user_ids: {len(test_user_ids)} users")
print(f"  test_targets: {len(test_targets)} users")

if len(test_user_ids) != len(test_targets):
    print("ERROR: test_user_ids and test_targets count mismatch!")
    alignment_ok = False

if alignment_ok:
    print("\nAlignment OK - index-based matching should work correctly")
else:
    print("\nWARNING: Alignment issues detected - results may be incorrect!")
    print("Consider using dialogue_id-based matching instead of index-based matching")

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================


def match_title_fuzzy(title: str, title_to_id: dict) -> int | None:
    """Match title with fuzzy fallback (same logic as notebook 06)."""
    # Direct match
    if title in title_to_id:
        return title_to_id[title]
    # Normalized match (lowercase, stripped)
    title_lower = title.strip().lower()
    for key, val in title_to_id.items():
        if key.lower() == title_lower:
            return val
    return None


def get_user_train_items(dataset) -> dict:
    """Get original item IDs each user interacted with in training (for masking).

    RecBole's full_sort_predict() returns RAW scores without masking.
    The trainer/evaluator applies masking AFTER getting scores.
    For our custom standalone evaluation, we must mask training items ourselves
    to avoid liked items appearing in top-K predictions.

    Returns: {original_user_id: set of original_item_ids}
    """
    uid_field = dataset.uid_field
    iid_field = dataset.iid_field
    uid_id2token = dataset.field2id_token[uid_field]
    iid_id2token = dataset.field2id_token[iid_field]

    user_train_items = {}
    for i in range(len(dataset.inter_feat[uid_field])):
        uid_token = uid_id2token[dataset.inter_feat[uid_field][i].item()]
        iid_token = iid_id2token[dataset.inter_feat[iid_field][i].item()]
        if uid_token == "[PAD]" or iid_token == "[PAD]":
            continue
        try:
            orig_uid = int(uid_token)
            orig_iid = int(iid_token)
        except ValueError:
            continue
        if orig_uid not in user_train_items:
            user_train_items[orig_uid] = set()
        user_train_items[orig_uid].add(orig_iid)
    return user_train_items


# =============================================================================
# LOAD CANDIDATE SETS
# =============================================================================

candidate_sets = {}  # {n_candidates: {user_id: [item_ids]}}

# Create title to item_id mapping (reverse of id_to_title)
title_to_id = {v: k for k, v in id_to_title.items()}

print("\n" + "=" * 60)
print("LOADING CANDIDATE SETS")
print("=" * 60)

# Load candidate files using CONTENT-BASED MATCHING (fingerprints)
# Load test templates for dialogue_id matching
test_templates = []
with open(DATA_PATH / "processed" / "test_prompt_templates.jsonl") as f:
    for line in f:
        test_templates.append(json.loads(line))
print(f"Loaded {len(test_templates)} test templates for dialogue_id matching")

unmatched_records = 0
for n_cand in CANDIDATE_SIZES:
    filepath = DATA_PATH / "processed" / f"test_prompt_examples_c{n_cand}_r10.jsonl"
    if not filepath.exists():
        print(f"WARNING: Candidate file not found: {filepath}")
        continue

    candidates = {}
    unmatched_titles = set()

    # Build JSONL idx → template idx mapping
    # Script 02 skips templates with no ground_truth, so JSONL records
    # correspond to non-skipped templates in order
    template_indices = []
    for ti, tmpl in enumerate(test_templates):
        if tmpl.get("recommended_accepted", []):
            template_indices.append(ti)

    with open(filepath) as f:
        for idx, line in enumerate(f):
            record = json.loads(line)

            # Content-based matching using dialogue_id from templates
            # Load dialogue_id from the corresponding template
            if idx >= len(template_indices):
                break
            template = test_templates[template_indices[idx]]
            did = str(template.get("dialogue_id", ""))
            user_id = dialogue_to_user.get(did)
            if user_id is None:
                unmatched_records += 1
                continue  # This dialogue was skipped in data preparation

            # Extract candidate titles from the prompt
            # The prompt has TWO <CANDIDATES> tags - use the second one (actual list)
            user_content = record["messages"][1]["content"]
            matches = list(re.finditer(r"<CANDIDATES>", user_content))

            if len(matches) >= 2:
                start = matches[1].start() + len("<CANDIDATES>")
                end_match = re.search(r"</CANDIDATES>|<[A-Z]", user_content[start:])
                if end_match:
                    end = start + end_match.start()
                else:
                    end = len(user_content)
                candidates_block = user_content[start:end]
                candidate_titles = [
                    line.strip()
                    for line in candidates_block.split("\n")
                    if line.strip()
                ]
            elif len(matches) == 1:
                start = matches[0].start() + len("<CANDIDATES>")
                end_match = re.search(r"</CANDIDATES>|<[A-Z]", user_content[start:])
                if end_match:
                    end = start + end_match.start()
                else:
                    end = len(user_content)
                candidates_block = user_content[start:end]
                candidate_titles = [
                    line.strip()
                    for line in candidates_block.split("\n")
                    if line.strip()
                ]
            else:
                candidate_titles = []

            # Convert to item IDs using fuzzy matching
            candidate_ids = []
            for title in candidate_titles:
                item_id = match_title_fuzzy(title, title_to_id)
                if item_id is not None:
                    candidate_ids.append(item_id)
                else:
                    unmatched_titles.add(title)

            if candidate_ids:
                candidates[user_id] = candidate_ids

    if candidates:
        candidate_sets[n_cand] = candidates
        print(f"Loaded {n_cand} candidates: {len(candidates)} users")
        if unmatched_titles:
            print(
                f"  Unmatched titles: {len(unmatched_titles)} (e.g., {list(unmatched_titles)[:3]})"
            )

if unmatched_records > 0:
    print(f"\nSkipped {unmatched_records} JSONL records with no matching RecBole user")
print(f"\nCandidate sets loaded: {list(candidate_sets.keys())}")


# =============================================================================
# METRIC COMPUTATION FUNCTIONS
# =============================================================================


def compute_filtered_metrics(
    model_scores: dict,
    candidate_sets: dict,
    test_targets: dict,
    id_to_title: dict,
    k_values: list = [1, 5, 10],
) -> dict:
    """Compute metrics on filtered candidate sets for fair LLM comparison."""
    results = {}

    for n_cand, user_candidates in candidate_sets.items():
        all_metrics = {
            k: {"hit": [], "mrr": [], "precision": [], "recall": [], "ndcg": []}
            for k in k_values
        }

        for user_id, candidate_ids in user_candidates.items():
            if user_id not in model_scores:
                continue
            if user_id not in test_targets:
                continue

            user_scores = model_scores[user_id]
            targets = test_targets[user_id]

            candidate_scores = []
            for item_id in candidate_ids:
                score = user_scores.get(item_id, float("-inf"))
                candidate_scores.append((item_id, score))

            candidate_scores.sort(key=lambda x: x[1], reverse=True)
            predictions = [item_id for item_id, _ in candidate_scores]

            target_titles = [id_to_title.get(tid, f"ID:{tid}") for tid in targets]
            pred_titles = [id_to_title.get(pid, f"ID:{pid}") for pid in predictions]

            metrics_dict = recommendation_metrics(
                predictions=pred_titles,
                ground_truth=target_titles,
                k_values=k_values,
            )

            for k in k_values:
                all_metrics[k]["hit"].append(metrics_dict[f"hit_rate@{k}"])
                all_metrics[k]["mrr"].append(metrics_dict[f"mrr@{k}"])
                all_metrics[k]["precision"].append(metrics_dict[f"precision@{k}"])
                all_metrics[k]["recall"].append(metrics_dict[f"recall@{k}"])
                all_metrics[k]["ndcg"].append(metrics_dict[f"ndcg@{k}"])

        results[n_cand] = {}
        for k in k_values:
            if all_metrics[k]["hit"]:
                results[n_cand][k] = {
                    "hit": np.mean(all_metrics[k]["hit"]),
                    "mrr": np.mean(all_metrics[k]["mrr"]),
                    "precision": np.mean(all_metrics[k]["precision"]),
                    "recall": np.mean(all_metrics[k]["recall"]),
                    "ndcg": np.mean(all_metrics[k]["ndcg"]),
                    "n_users": len(all_metrics[k]["hit"]),
                }

    return results


def compute_filtered_metrics_detailed(
    model_name: str,
    model_scores: dict,
    candidate_sets: dict,
    test_targets: dict,
    id_to_title: dict,
    k_values: list = [1, 5, 10],
) -> tuple[dict, list]:
    """Compute metrics with per-user detail for CSV export matching LLM format."""
    aggregated = {}
    per_user_records = []

    for n_cand, user_candidates in candidate_sets.items():
        all_metrics = {
            k: {"hit": [], "mrr": [], "precision": [], "recall": [], "ndcg": []}
            for k in k_values
        }

        for user_id, candidate_ids in user_candidates.items():
            if user_id not in model_scores:
                continue
            if user_id not in test_targets:
                continue

            user_scores = model_scores[user_id]
            targets = test_targets[user_id]

            candidate_scores = []
            for item_id in candidate_ids:
                score = user_scores.get(item_id, float("-inf"))
                candidate_scores.append((item_id, score))

            candidate_scores.sort(key=lambda x: x[1], reverse=True)
            predictions = [item_id for item_id, _ in candidate_scores]

            target_titles = [id_to_title.get(tid, f"ID:{tid}") for tid in targets]
            pred_titles = [
                id_to_title.get(pid, f"ID:{pid}") for pid in predictions[:10]
            ]
            all_pred_titles = [id_to_title.get(pid, f"ID:{pid}") for pid in predictions]

            metrics_dict = recommendation_metrics(
                predictions=all_pred_titles,
                ground_truth=target_titles,
                k_values=k_values,
            )

            metrics_by_k = {}
            for k in k_values:
                metrics_by_k[k] = {
                    "hit": metrics_dict[f"hit_rate@{k}"],
                    "mrr": metrics_dict[f"mrr@{k}"],
                    "precision": metrics_dict[f"precision@{k}"],
                    "recall": metrics_dict[f"recall@{k}"],
                    "f1": metrics_dict[f"f1@{k}"],
                    "ndcg": metrics_dict[f"ndcg@{k}"],
                }
                all_metrics[k]["hit"].append(metrics_by_k[k]["hit"])
                all_metrics[k]["mrr"].append(metrics_by_k[k]["mrr"])
                all_metrics[k]["precision"].append(metrics_by_k[k]["precision"])
                all_metrics[k]["recall"].append(metrics_by_k[k]["recall"])
                all_metrics[k]["ndcg"].append(metrics_by_k[k]["ndcg"])

            record = {
                "model": model_name,  # Raw name; CBF+ prefix added in comparison DataFrame
                "model_type": "CF",
                "model_id": model_name,
                "n_candidates": f"c{n_cand}",
                "prompt_idx": -1,
                "response": "",
                "pred_items": pred_titles,
                "num_pred_items": len(pred_titles),
                "ground_truth": target_titles,
                "num_ground_truth": len(targets),
                "entropy": 0.0,
                "normalized_entropy": 0.0,
                "unique_token_ratio": 0.0,
            }
            for k in k_values:
                record[f"hit_rate@{k}"] = metrics_by_k[k]["hit"]
                record[f"mrr@{k}"] = metrics_by_k[k]["mrr"]
                record[f"precision@{k}"] = metrics_by_k[k]["precision"]
                record[f"recall@{k}"] = metrics_by_k[k]["recall"]
                record[f"f1@{k}"] = metrics_by_k[k]["f1"]
                record[f"ndcg@{k}"] = metrics_by_k[k]["ndcg"]

            per_user_records.append(record)

        aggregated[n_cand] = {}
        for k in k_values:
            if all_metrics[k]["hit"]:
                aggregated[n_cand][k] = {
                    "hit": np.mean(all_metrics[k]["hit"]),
                    "mrr": np.mean(all_metrics[k]["mrr"]),
                    "precision": np.mean(all_metrics[k]["precision"]),
                    "recall": np.mean(all_metrics[k]["recall"]),
                    "ndcg": np.mean(all_metrics[k]["ndcg"]),
                    "n_users": len(all_metrics[k]["hit"]),
                }

    return aggregated, per_user_records


# =============================================================================
# MODEL SCORING FUNCTION (WITH PROPER ID MAPPING)
# =============================================================================


def get_model_item_scores(model, dataset, test_users: list) -> dict:
    """Get item scores from trained model for all test users.

    IMPORTANT: This function handles RecBole's internal ID remapping.
    - RecBole remaps token fields (user_id, item_id) to internal contiguous IDs
    - We must convert original user IDs -> internal IDs for model input
    - We must convert internal item IDs -> original IDs for score output

    Returns: {original_user_id: {original_item_id: score}}
    """
    model.eval()
    scores = {}

    user_token2id = dataset.field2token_id[dataset.uid_field]
    item_id2token = dataset.field2id_token[dataset.iid_field]

    n_items = dataset.item_num

    print("  RecBole ID mapping info:")
    print(f"    Total users in dataset: {dataset.user_num}")
    print(f"    Total items in dataset: {n_items}")
    print(f"    Sample user mapping: {list(user_token2id.items())[:3]}...")
    print(
        f"    Sample item mapping: item_id2token[1] = {item_id2token[1] if len(item_id2token) > 1 else 'N/A'}"
    )

    missing_users = 0
    error_count = 0
    score_offset = None
    with torch.no_grad():
        for original_user_id in tqdm(
            test_users, desc="Getting user scores", leave=False
        ):
            internal_user_id = user_token2id.get(str(original_user_id))

            if internal_user_id is None:
                missing_users += 1
                continue

            try:
                user_tensor = torch.LongTensor([internal_user_id]).to(model.device)

                # Try full_sort_predict first (most models), fall back to
                # pair-wise predict (e.g. NeuMF raises NotImplementedError)
                from recbole.data.interaction import Interaction

                try:
                    interaction = Interaction({dataset.uid_field: user_tensor})
                    interaction = interaction.to(model.device)
                    item_scores = model.full_sort_predict(interaction)
                    item_scores = item_scores.cpu().numpy().flatten()
                except NotImplementedError:
                    item_tensor = torch.arange(n_items).to(model.device)
                    user_tensor_expanded = user_tensor.expand(n_items)
                    interaction = Interaction(
                        {
                            dataset.uid_field: user_tensor_expanded,
                            dataset.iid_field: item_tensor,
                        }
                    )
                    interaction = interaction.to(model.device)
                    item_scores = model.predict(interaction).cpu().numpy()

                if score_offset is None:
                    token_len = len(item_id2token)
                    score_len = len(item_scores)
                    if score_len == token_len:
                        score_offset = 0
                    elif score_len == token_len - 1:
                        score_offset = 1
                    else:
                        score_offset = 0
                        print(
                            f"  WARNING: score length {score_len} does not match token length {token_len}; using offset=0"
                        )
                    print(
                        f"  Score vector length: {score_len}, item_id2token length: {token_len}, offset: {score_offset}"
                    )

                user_score_dict = {}
                for internal_item_id, score in enumerate(item_scores):
                    token_index = internal_item_id + score_offset
                    if token_index >= len(item_id2token):
                        break
                    original_item_token = item_id2token[token_index]
                    if original_item_token != "[PAD]":
                        try:
                            original_item_id = int(original_item_token)
                            user_score_dict[original_item_id] = float(score)
                        except ValueError:
                            continue

                scores[original_user_id] = user_score_dict

            except Exception as e:
                error_count += 1
                if error_count <= 3:
                    print(
                        f"  Error getting scores for user {original_user_id}: {type(e).__name__}({e!r})"
                    )
                continue

    if missing_users > 0:
        print(f"  WARNING: {missing_users} users not found in RecBole dataset")
    if error_count > 0:
        print(f"  WARNING: {error_count} users failed during scoring")

    print(f"  Successfully scored {len(scores)} users")

    return scores


print("\nEvaluation functions defined (with proper ID mapping)")

In [ ]:
# Candidate coverage sanity check
print("\n" + "=" * 60)
print("CANDIDATE COVERAGE")
print("=" * 60)
for n_cand, user_candidates in candidate_sets.items():
    users_with_target = 0
    for uid, cand in user_candidates.items():
        targets = test_targets.get(uid, [])
        if any(t in cand for t in targets):
            users_with_target += 1
    coverage = users_with_target / len(test_user_ids) if test_user_ids else 0
    print(
        f"  {n_cand} candidates: target covered for {users_with_target}/{len(test_user_ids)} users ({coverage:.3f})"
    )
print("Note: coverage is an upper bound on Hit@k for that candidate size.")

In [ ]:
# Run standalone (full-set) and CBF+reranker (filtered) evaluation
# Uses recommendation_metrics() for both, ensuring consistent comparison

filtered_results = {}  # {model_name: {n_candidates: {k: {metrics}}}}
standalone_results = {}  # {model_name: {k: {metrics}}}
all_per_user_records = []  # Per-user records for CSV export

print("Running evaluation for all successfully trained models...")

for model_name in tqdm(ALL_MODELS, desc="Evaluating models"):
    result = recbole_results.get(model_name, {})
    if result.get("status") != "success":
        print(f"\n  Skipping {model_name} (training failed)")
        continue

    print(f"\n{'=' * 60}")
    print(f"Evaluating: {model_name}")
    print(f"{'=' * 60}")

    try:
        model = result["model"]
        dataset = result["dataset"]

        # Get item scores for all test users (used for both standalone and filtered)
        model_scores = get_model_item_scores(model, dataset, test_user_ids)

        # --- Standalone evaluation (full catalog, with training-item masking) ---
        # RecBole's full_sort_predict() returns RAW scores without masking.
        # We must exclude training items (liked items) to match RecBole's internal eval.
        user_train_items = get_user_train_items(dataset)
        standalone_metrics = {
            k: {"hit": [], "mrr": [], "precision": [], "recall": [], "ndcg": []}
            for k in K_VALUES
        }

        for user_id in test_user_ids:
            if user_id not in model_scores or user_id not in test_targets:
                continue

            user_scores = model_scores[user_id]
            targets = test_targets[user_id]

            # Rank items by score, excluding training items (liked items)
            train_items = user_train_items.get(user_id, set())
            sorted_items = sorted(
                (
                    (iid, score)
                    for iid, score in user_scores.items()
                    if iid not in train_items
                ),
                key=lambda x: x[1],
                reverse=True,
            )
            all_pred_titles = [
                id_to_title.get(iid, f"ID:{iid}") for iid, _ in sorted_items
            ]
            target_titles = [id_to_title.get(tid, f"ID:{tid}") for tid in targets]

            metrics_dict = recommendation_metrics(
                predictions=all_pred_titles,
                ground_truth=target_titles,
                k_values=K_VALUES,
            )

            for k in K_VALUES:
                standalone_metrics[k]["hit"].append(metrics_dict[f"hit_rate@{k}"])
                standalone_metrics[k]["mrr"].append(metrics_dict[f"mrr@{k}"])
                standalone_metrics[k]["precision"].append(
                    metrics_dict[f"precision@{k}"]
                )
                standalone_metrics[k]["recall"].append(metrics_dict[f"recall@{k}"])
                standalone_metrics[k]["ndcg"].append(metrics_dict[f"ndcg@{k}"])

        standalone_results[model_name] = {}
        for k in K_VALUES:
            if standalone_metrics[k]["hit"]:
                standalone_results[model_name][k] = {
                    "hit": np.mean(standalone_metrics[k]["hit"]),
                    "mrr": np.mean(standalone_metrics[k]["mrr"]),
                    "precision": np.mean(standalone_metrics[k]["precision"]),
                    "recall": np.mean(standalone_metrics[k]["recall"]),
                    "ndcg": np.mean(standalone_metrics[k]["ndcg"]),
                    "n_users": len(standalone_metrics[k]["hit"]),
                }

        if 10 in standalone_results[model_name]:
            m = standalone_results[model_name][10]
            print(f"  Standalone: NDCG@10={m['ndcg']:.4f}, Hit@10={m['hit']:.4f}")

        # --- CBF+Reranker evaluation (filtered candidates) ---
        if candidate_sets:
            model_filtered, per_user_records = compute_filtered_metrics_detailed(
                model_name=model_name,
                model_scores=model_scores,
                candidate_sets=candidate_sets,
                test_targets=test_targets,
                id_to_title=id_to_title,
                k_values=K_VALUES,
            )

            filtered_results[model_name] = model_filtered
            all_per_user_records.extend(per_user_records)

            if 250 in model_filtered:
                m = model_filtered[250].get(10, {})
                print(
                    f"  CBF+{model_name} (250): NDCG@10={m.get('ndcg', 0):.4f}, Hit@10={m.get('hit', 0):.4f}"
                )

    except Exception as e:
        print(f"  Error evaluating {model_name}: {e}")
        import traceback

        traceback.print_exc()
        continue

print("\nEvaluation complete:")
print(f"  Standalone: {len(standalone_results)} models")
print(f"  CBF+Reranker: {len(filtered_results)} models")
print(f"  Per-user records: {len(all_per_user_records)}")

## Load LLM Results

In [ ]:
# Check if LLM results exist
parquet_files = sorted(
    p
    for p in (DATA_PATH / "output").glob("evaluation_results_*.parquet")
    if "recbole" not in p.name  # Exclude RecBole outputs, keep only LLM results
)

if not parquet_files:
    print("WARNING: No LLM evaluation parquet files found in data/output/")
    print("Please run scripts/04_evaluation.ipynb first.")
    df_llm = None
else:
    df_llm = pd.concat([pd.read_parquet(p) for p in parquet_files], ignore_index=True)
    print(f"Loaded {len(parquet_files)} LLM parquet files:")
    for p in parquet_files:
        print(f"  - {Path(p).name}")
    print(f"\nTotal: {len(df_llm)} LLM evaluation records")
    print(f"Models: {sorted(df_llm['model'].unique())}")
    print(f"Candidate sizes: {sorted(df_llm['n_candidates'].unique())}")

## Compile Comparison Results

In [ ]:
# Create comparison DataFrame with method-based naming
comparison_records = []

# Add standalone CF results (custom metrics from recommendation_metrics)
for model_name, k_results in standalone_results.items():
    for k, metrics in k_results.items():
        comparison_records.append(
            {
                "model_type": "CF",
                "model": model_name,
                "eval_method": "standalone",
                "n_candidates": 6924,
                "k": k,
                "hit_rate": metrics.get("hit", np.nan),
                "mrr": metrics.get("mrr", np.nan),
                "precision": metrics.get("precision", np.nan),
                "recall": metrics.get("recall", np.nan),
                "ndcg": metrics.get("ndcg", np.nan),
            }
        )

# Add CBF+CF reranker results
for model_name, model_filtered in filtered_results.items():
    for n_cand, k_results in model_filtered.items():
        for k, metrics in k_results.items():
            comparison_records.append(
                {
                    "model_type": "CF",
                    "model": f"CBF+{model_name}",
                    "eval_method": "cbf_reranking",
                    "n_candidates": n_cand,
                    "k": k,
                    "hit_rate": metrics.get("hit", np.nan),
                    "mrr": metrics.get("mrr", np.nan),
                    "precision": metrics.get("precision", np.nan),
                    "recall": metrics.get("recall", np.nan),
                    "ndcg": metrics.get("ndcg", np.nan),
                }
            )


# 2b. Add LLM standalone results (cALL = full catalog, comparable to CF/Seq standalone)
if df_llm is not None:
    df_llm_all = df_llm[df_llm["n_candidates"] == "cALL"]
    if not df_llm_all.empty:
        for model in df_llm_all["model"].unique():
            model_df = df_llm_all[df_llm_all["model"] == model]
            for k in K_VALUES:
                comparison_records.append(
                    {
                        "model_type": "LLM",
                        "model": model,
                        "eval_method": "standalone",
                        "n_candidates": len(id_to_title),
                        "k": k,
                        "hit_rate": model_df[f"hit_rate@{k}"].mean(),
                        "mrr": model_df[f"mrr@{k}"].mean(),
                        "precision": model_df[f"precision@{k}"].mean(),
                        "recall": model_df[f"recall@{k}"].mean(),
                        "ndcg": model_df[f"ndcg@{k}"].mean(),
                    }
                )
        print(f"Added {len(df_llm_all['model'].unique())} LLM standalone models (cALL)")

    # 2c. Add LLM zero-shot results (c0 = no candidates, pure LLM knowledge)
    df_llm_zs = df_llm[df_llm["n_candidates"] == "c0"]
    if not df_llm_zs.empty:
        for model in df_llm_zs["model"].unique():
            model_df = df_llm_zs[df_llm_zs["model"] == model]
            for k in K_VALUES:
                comparison_records.append(
                    {
                        "model_type": "LLM",
                        "model": model,
                        "eval_method": "zero_shot",
                        "n_candidates": 0,
                        "k": k,
                        "hit_rate": model_df[f"hit_rate@{k}"].mean(),
                        "mrr": model_df[f"mrr@{k}"].mean(),
                        "precision": model_df[f"precision@{k}"].mean(),
                        "recall": model_df[f"recall@{k}"].mean(),
                        "ndcg": model_df[f"ndcg@{k}"].mean(),
                    }
                )
        print(f"Added {len(df_llm_zs['model'].unique())} LLM zero-shot models (c0)")

# Add LLM results (always CBF+reranker)
if df_llm is not None:
    for n_cand in CANDIDATE_SIZES:
        df_llm_subset = df_llm[(df_llm["n_candidates"] == f"c{n_cand}")]

        if df_llm_subset.empty:
            continue

        for model in df_llm_subset["model"].unique():
            model_df = df_llm_subset[df_llm_subset["model"] == model]
            for k in K_VALUES:
                comparison_records.append(
                    {
                        "model_type": "LLM",
                        "model": f"CBF+{model}",
                        "eval_method": "cbf_reranking",
                        "n_candidates": n_cand,
                        "k": k,
                        "hit_rate": model_df[f"hit_rate@{k}"].mean(),
                        "mrr": model_df[f"mrr@{k}"].mean(),
                        "precision": model_df[f"precision@{k}"].mean(),
                        "recall": model_df[f"recall@{k}"].mean(),
                        "ndcg": model_df[f"ndcg@{k}"].mean(),
                    }
                )

df_comparison = pd.DataFrame.from_records(comparison_records)
print(f"Created comparison DataFrame with {len(df_comparison)} rows")
print(f"Evaluation methods: {df_comparison['eval_method'].unique()}")
print(f"Models: {df_comparison['model'].unique()}")

## Methodology Documentation

### Method Categories

| Category | Models | Candidate Pool |
|----------|--------|---------------|
| **Standalone** | CF models (Pop, ItemKNN, LightGCN, ...) | Full catalog (~6,924 items) |
| **CBF+Reranker** | CF rerankers (CBF+Pop, CBF+ItemKNN, ...) | 100-1000 CBF-retrieved candidates |
| **CBF+Reranker** | LLM rerankers (CBF+GPT-4.1, CBF+LLaMA-3-8B) | 100-1000 CBF-retrieved candidates |

### Metric Computation

All methods use the same `recommendation_metrics()` function from `src/stability/metrics.py`.
This ensures direct comparability across all standalone and CBF+reranker methods.

## Visualization

In [ ]:
# Visualization 1: Standalone CF vs CBF+Reranker comparison
from matplotlib.patches import Patch

# All models at K=10: standalone + CBF+reranker at 250
df_standalone_k10 = df_comparison[
    (df_comparison["eval_method"] == "standalone") & (df_comparison["k"] == 10)
]
df_cbf_250_k10 = df_comparison[
    (df_comparison["eval_method"] == "cbf_reranking")
    & (df_comparison["n_candidates"] == 250)
    & (df_comparison["k"] == 10)
]

df_combined = pd.concat([df_standalone_k10, df_cbf_250_k10])

if len(df_combined) > 0:
    n_models = len(df_combined)
    fig_height = max(8, n_models * 0.6 + 2)
    fig, axes = plt.subplots(2, 2, figsize=(16, fig_height))
    fig.suptitle(
        "All Methods: Standalone (6,924 items) vs CBF+Reranker (250 candidates)",
        fontsize=14,
        fontweight="bold",
    )

    metrics = ["ndcg", "mrr", "hit_rate", "recall"]
    titles = ["NDCG@10", "MRR@10", "Hit Rate@10", "Recall@10"]

    type_colors = {"CF": "steelblue", "LLM": "forestgreen"}

    for ax, metric, title in zip(axes.flat, metrics, titles):
        df_sorted = df_combined.sort_values(metric, ascending=True)
        # Color by: standalone CF = steelblue, CBF+CF = lightskyblue, CBF+LLM = forestgreen
        colors = []
        for _, row in df_sorted.iterrows():
            if row["model_type"] == "LLM":
                colors.append("forestgreen")
            elif row["eval_method"] == "standalone":
                colors.append("steelblue")
            else:
                colors.append("lightskyblue")

        ax.barh(df_sorted["model"], df_sorted[metric], color=colors)
        ax.set_xlabel(title)
        ax.set_title(title)

        for i, (val, model) in enumerate(zip(df_sorted[metric], df_sorted["model"])):
            ax.text(val + 0.002, i, f"{val:.3f}", va="center", fontsize=8)

    legend_elements = [
        Patch(facecolor="steelblue", label="Standalone CF (6,924 items)"),
        Patch(facecolor="lightskyblue", label="CBF+CF Reranker (250 items)"),
        Patch(facecolor="forestgreen", label="CBF+LLM Reranker (250 items)"),
    ]
    fig.legend(
        handles=legend_elements, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02)
    )

    plt.tight_layout()
    plt.savefig(
        RESULTS_PATH / "comparison_all_methods.png", dpi=150, bbox_inches="tight"
    )
    plt.show()
    print(f"Saved to {RESULTS_PATH / 'comparison_all_methods.png'}")

In [ ]:
# Visualization 2: CBF+Reranker comparison only (250 candidates)
# All rerankers on the same candidate pool

df_cbf_250_k10 = df_comparison[
    (df_comparison["eval_method"] == "cbf_reranking")
    & (df_comparison["n_candidates"] == 250)
    & (df_comparison["k"] == 10)
].copy()

if len(df_cbf_250_k10) > 0:
    n_models = len(df_cbf_250_k10)
    fig_height = max(8, n_models * 0.6 + 2)
    fig, axes = plt.subplots(2, 2, figsize=(14, fig_height))
    fig.suptitle(
        "CBF+Reranker Comparison: 250 candidates (same task for all)",
        fontsize=14,
        fontweight="bold",
    )

    metrics = ["ndcg", "mrr", "hit_rate", "recall"]
    titles = ["NDCG@10", "MRR@10", "Hit Rate@10", "Recall@10"]

    for ax, metric, title in zip(axes.flat, metrics, titles):
        df_sorted = df_cbf_250_k10.sort_values(metric, ascending=True)
        colors = df_sorted["model_type"].map(
            {"CF": "lightskyblue", "LLM": "forestgreen"}
        )

        ax.barh(df_sorted["model"], df_sorted[metric], color=colors)
        ax.set_xlabel(title)
        ax.set_title(title)

        for i, (val, model) in enumerate(zip(df_sorted[metric], df_sorted["model"])):
            ax.text(val + 0.002, i, f"{val:.3f}", va="center", fontsize=8)

    legend_elements = [
        Patch(facecolor="lightskyblue", label="CBF+CF Reranker"),
        Patch(facecolor="forestgreen", label="CBF+LLM Reranker"),
    ]
    fig.legend(
        handles=legend_elements, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02)
    )

    plt.tight_layout()
    plt.savefig(
        RESULTS_PATH / "comparison_cbf_rerankers_250.png", dpi=150, bbox_inches="tight"
    )
    plt.show()
    print(f"Saved to {RESULTS_PATH / 'comparison_cbf_rerankers_250.png'}")

In [ ]:
# Visualization 3: NDCG@10 vs candidate pool size (full spectrum)
# Include: zero-shot (0) -> CBF+Reranker (100-1000) -> standalone (all items)

N_ITEMS = 6924  # full catalog size

# Gather all relevant data at k=10
df_plot3 = df_comparison[
    (df_comparison["k"] == 10)
    & (df_comparison["eval_method"].isin(["cbf_reranking", "standalone", "zero_shot"]))
].copy()

if len(df_plot3) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))

    # Strip CBF+ prefix to unify lines (e.g., CBF+ItemKNN and ItemKNN are the same model)
    df_plot3["base_model"] = df_plot3["model"].str.replace("CBF+", "", regex=False)

    for base_model in df_plot3["base_model"].unique():
        model_data = df_plot3[df_plot3["base_model"] == base_model].copy()
        # Replace 0 with 1 for log scale (will label as 0)
        model_data["x"] = model_data["n_candidates"].replace(0, 1)
        model_data = model_data.sort_values("x")
        model_type = model_data["model_type"].iloc[0]
        linestyle = "--" if model_type == "LLM" else "-"
        marker = "s" if model_type == "LLM" else "o"
        linewidth = 2 if model_type == "LLM" else 1.5
        ax.plot(
            model_data["x"],
            model_data["ndcg"],
            marker=marker,
            linestyle=linestyle,
            linewidth=linewidth,
            label=base_model,
        )

    ax.set_xlabel("Number of Candidates")
    ax.set_ylabel("NDCG@10")
    ax.set_title("NDCG@10 vs Candidate Pool Size: Zero-Shot to Full Catalog")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xscale("log")
    tick_positions = [1, 100, 250, 500, 1000, N_ITEMS]
    tick_labels = ["0", "100", "250", "500", "1000", f"{N_ITEMS:,}"]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)

    plt.tight_layout()
    plt.savefig(RESULTS_PATH / "ndcg_vs_candidates.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {RESULTS_PATH / 'ndcg_vs_candidates.png'}")

## Summary Statistics

In [ ]:
# Summary tables at K=10

# Table 1: Standalone CF results (full catalog)
print("\n" + "=" * 80)
print("STANDALONE CF MODELS (K=10, ~6,924 items)")
print("=" * 80)
df_standalone = df_comparison[
    (df_comparison["eval_method"] == "standalone") & (df_comparison["k"] == 10)
].sort_values("ndcg", ascending=False)

if len(df_standalone) > 0:
    display_cols = ["model", "ndcg", "mrr", "hit_rate", "precision", "recall"]
    df_display = df_standalone[display_cols].copy()
    for col in ["ndcg", "mrr", "hit_rate", "precision", "recall"]:
        df_display[col] = df_display[col].map(lambda x: f"{x:.4f}")
    print(df_display.to_string(index=False))

# Table 2: CBF+Reranker at 250 candidates
print("\n" + "=" * 80)
print("CBF+RERANKER COMPARISON: 250 CANDIDATES (K=10)")
print("=" * 80)
df_cbf = df_comparison[
    (df_comparison["eval_method"] == "cbf_reranking")
    & (df_comparison["n_candidates"] == 250)
    & (df_comparison["k"] == 10)
].sort_values("ndcg", ascending=False)

if len(df_cbf) > 0:
    display_cols = [
        "model_type",
        "model",
        "ndcg",
        "mrr",
        "hit_rate",
        "precision",
        "recall",
    ]
    df_display = df_cbf[display_cols].copy()
    for col in ["ndcg", "mrr", "hit_rate", "precision", "recall"]:
        df_display[col] = df_display[col].map(lambda x: f"{x:.4f}")
    print(df_display.to_string(index=False))

In [ ]:
# Best models by category
print("\n" + "=" * 60)
print("BEST MODELS BY CATEGORY (NDCG@10)")
print("=" * 60)

df_k10 = df_comparison[df_comparison["k"] == 10]

# Best standalone CF
df_sa = df_k10[df_k10["eval_method"] == "standalone"]
if len(df_sa) > 0:
    best = df_sa.loc[df_sa["ndcg"].idxmax()]
    print("\nBest Standalone CF (~6,924 items):")
    print(
        f"  {best['model']}: NDCG={best['ndcg']:.4f}, MRR={best['mrr']:.4f}, Hit={best['hit_rate']:.4f}"
    )

# Best CBF+CF reranker at 250
df_cbf_cf = df_k10[
    (df_k10["eval_method"] == "cbf_reranking")
    & (df_k10["n_candidates"] == 250)
    & (df_k10["model_type"] == "CF")
]
if len(df_cbf_cf) > 0:
    best = df_cbf_cf.loc[df_cbf_cf["ndcg"].idxmax()]
    print("\nBest CBF+CF Reranker (250 candidates):")
    print(
        f"  {best['model']}: NDCG={best['ndcg']:.4f}, MRR={best['mrr']:.4f}, Hit={best['hit_rate']:.4f}"
    )

# Best CBF+LLM reranker at 250
df_cbf_llm = df_k10[
    (df_k10["eval_method"] == "cbf_reranking")
    & (df_k10["n_candidates"] == 250)
    & (df_k10["model_type"] == "LLM")
]
if len(df_cbf_llm) > 0:
    best = df_cbf_llm.loc[df_cbf_llm["ndcg"].idxmax()]
    print("\nBest CBF+LLM Reranker (250 candidates):")
    print(
        f"  {best['model']}: NDCG={best['ndcg']:.4f}, MRR={best['mrr']:.4f}, Hit={best['hit_rate']:.4f}"
    )

# Cross-category summary
if len(df_cbf_cf) > 0 and len(df_cbf_llm) > 0:
    best_cf = df_cbf_cf.loc[df_cbf_cf["ndcg"].idxmax()]
    best_llm = df_cbf_llm.loc[df_cbf_llm["ndcg"].idxmax()]

    print("\n" + "=" * 60)
    print("RERANKER COMPARISON (250 candidates)")
    print("=" * 60)
    print(f"Best CBF+CF  ({best_cf['model']}): NDCG@10 = {best_cf['ndcg']:.4f}")
    print(f"Best CBF+LLM ({best_llm['model']}): NDCG@10 = {best_llm['ndcg']:.4f}")

    if best_llm["ndcg"] > best_cf["ndcg"] and best_cf["ndcg"] > 0:
        ratio = best_llm["ndcg"] / best_cf["ndcg"]
        print(
            f"\nLLM reranker outperforms best CF reranker by {ratio:.1f}x on same candidate set"
        )

## Save Results

In [ ]:
# Save comparison DataFrame
df_comparison.to_csv(RESULTS_PATH / "recbole_llm_comparison.csv", index=False)
print(f"Saved comparison to {RESULTS_PATH / 'recbole_llm_comparison.csv'}")

# Create and save summary (K=10 results)
df_summary = df_comparison[df_comparison["k"] == 10].copy()
df_summary.to_csv(RESULTS_PATH / "comparison_summary.csv", index=False)
print(f"Saved summary to {RESULTS_PATH / 'comparison_summary.csv'}")

# Save RecBole results
recbole_export = {}
for model_name, results in recbole_results.items():
    if results["status"] == "success":
        recbole_export[model_name] = {
            "test_result": results["test_result"],
            "best_params": best_hyperparams.get(model_name, {}).get("params", {}),
        }

with open(RESULTS_PATH / "recbole_final_results.json", "w") as f:
    json.dump(recbole_export, f, indent=2)
print(f"Saved RecBole results to {RESULTS_PATH / 'recbole_final_results.json'}")

In [ ]:
# Save per-user results in LLM-compatible format
# This enables direct comparison with evaluation_results_full.csv

if all_per_user_records:
    df_traditional = pd.DataFrame.from_records(all_per_user_records)

    # Reorder columns to match NB04 LLM parquet schema
    column_order = [
        "model",
        "model_type",
        "model_id",
        "n_candidates",
        "prompt_idx",
        "response",
        "pred_items",
        "num_pred_items",
        "ground_truth",
        "num_ground_truth",
        "entropy",
        "normalized_entropy",
        "unique_token_ratio",
        "hit_rate@1",
        "mrr@1",
        "precision@1",
        "recall@1",
        "f1@1",
        "ndcg@1",
        "hit_rate@5",
        "mrr@5",
        "precision@5",
        "recall@5",
        "f1@5",
        "ndcg@5",
        "hit_rate@10",
        "mrr@10",
        "precision@10",
        "recall@10",
        "f1@10",
        "ndcg@10",
    ]
    df_traditional = df_traditional[column_order]

    # Save as parquet (same format as NB04 LLM results)
    output_path = DATA_PATH / "output" / "evaluation_results_recbole_cf.parquet"
    df_traditional.to_parquet(output_path, index=False)

    print(f"\nSaved {len(df_traditional)} per-user results to {output_path}")
    print(f"Models: {df_traditional['model'].unique().tolist()}")
    print(f"Candidate sizes: {sorted(df_traditional['n_candidates'].unique())}")
    print("\nFormat matches NB04 parquet schema for direct comparison")
else:
    print("No per-user records to save (filtered evaluation was skipped)")

## Key Findings

In [ ]:
print("\n" + "=" * 60)
print("KEY FINDINGS")
print("=" * 60)

df_k10 = df_comparison[df_comparison["k"] == 10]

# 1. Best standalone CF
df_sa = df_k10[df_k10["eval_method"] == "standalone"]
if len(df_sa) > 0:
    best = df_sa.loc[df_sa["ndcg"].idxmax()]
    print("\n1. Best Standalone CF (~6,924 items):")
    print(f"   {best['model']}: NDCG@10 = {best['ndcg']:.4f}")

# 2. Reranker comparison at 250 candidates
df_cbf_250 = df_k10[
    (df_k10["eval_method"] == "cbf_reranking") & (df_k10["n_candidates"] == 250)
]
if len(df_cbf_250) > 0:
    df_cf_rerank = df_cbf_250[df_cbf_250["model_type"] == "CF"]
    df_llm_rerank = df_cbf_250[df_cbf_250["model_type"] == "LLM"]

    if len(df_cf_rerank) > 0 and len(df_llm_rerank) > 0:
        best_cf = df_cf_rerank.loc[df_cf_rerank["ndcg"].idxmax()]
        best_llm = df_llm_rerank.loc[df_llm_rerank["ndcg"].idxmax()]

        print("\n2. CBF+Reranker Comparison (250 candidates):")
        print(f"   Best CBF+CF  ({best_cf['model']}): NDCG@10 = {best_cf['ndcg']:.4f}")
        print(
            f"   Best CBF+LLM ({best_llm['model']}): NDCG@10 = {best_llm['ndcg']:.4f}"
        )

        if best_llm["ndcg"] > best_cf["ndcg"] and best_cf["ndcg"] > 0:
            ratio = best_llm["ndcg"] / best_cf["ndcg"]
            print("\n3. Key Insight:")
            print(f"   LLM reranker outperforms best CF reranker by {ratio:.1f}x")
            print("   on the same CBF-retrieved candidate set")

print("\n4. Output Files:")
print(f"   - {RESULTS_PATH / 'recbole_llm_comparison.csv'}")
print(f"   - {RESULTS_PATH / 'comparison_all_methods.png'}")
print(f"   - {RESULTS_PATH / 'comparison_cbf_rerankers_250.png'}")
print(f"   - {RESULTS_PATH / 'ndcg_vs_candidates.png'}")

In [ ]:
print("\nEvaluation complete!")